In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import nltk
from nltk.corpus import stopwords

In [ ]:
# Only needed once
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
data = pd.read_csv('995,000_rows.csv')
data.head()

## Clean the Data

In [ ]:
def clean_text(text):
    # 1. Håndter tomme værdier (vigtigt for det store datasæt)
    if pd.isna(text):
        return ""
    
    # 2. Gør alt til små bogstaver
    text = text.lower()

    # 3. Erstat URLs med et tag (bevarer information om at der var et link)
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)

    # 4. Erstat e-mails med et tag
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '<EMAIL>', text)

    # 5. Erstat datoer (f.eks. 13th) med et tag
    text = re.sub(r'[0-9]+[a-zA-Z]+', '<DATE>', text)

    # 6. Erstat resterende tal med et tag
    text = re.sub(r'[0-9]+', '<NUM>', text)
    
    # 7. Fjern specialtegn (behold kun bogstaver, tags og mellemrum)
    text = re.sub(r'[^a-z\s<>]', '', text)
    
    # 8. Fjern ekstra mellemrum, tabs og linjeskift
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

data["clean_content"] = data["content"].apply(clean_text)
data[["content", "clean_content"]].head()

In [ ]:
data['tokens'] = data['clean_content'].apply(nltk.word_tokenize)
data[['clean_content', 'tokens']].head()

In [ ]:
# Store English stopwords into a variable
english_stopwords = set(stopwords.words('english'))
print(f"Number of stopwords: {len(english_stopwords)}")
print(list(english_stopwords)[:15]) # Display first 15 stopwords

data['filtered_tokens'] = data['tokens'].apply(lambda tokens: [word for word in tokens if word not in english_stopwords])
data[['tokens', 'filtered_tokens']].head()

In [ ]:
# Liste over alle ord i alle artikler
#all_words_list = [word for tokens_list in data['tokens'] for word in tokens_list]

# Sæt af unikke ord (vocab) og beregn vocab size
#vocab = set(all_words_list)

# Længden af vocab
#vocab_size = len(vocab)

#print(f"Antal tokens i alt (alle ord i alle artikler): {len(all_words_list)}")
#print(f"Vocabulary size (unikke ord): {vocab_size}")
#print(f"10 ord i the vocab (bare lige for at se): {list(vocab)[:10]}")

In [ ]:
# Liste over alle ord i alle artikler
#filtered_words_list = [word for tokens_list in data['filtered_tokens'] for word in tokens_list]

# Sæt af unikke ord (vocab) og beregn vocab size
#vocab_filtered = set(filtered_words_list)

# Længden af vocab
#filtered_vocab_size = len(vocab_filtered)

#print(f"Antal tokens i alt (alle ord i alle artikler): {len(filtered_words_list)}")
#print(f"Vocabulary size (unikke ord): {filtered_vocab_size}")
#print(f"10 ord i the vocab (bare lige for at se): {list(vocab_filtered)[:10]}")

In [ ]:
ps = PorterStemmer()

# Anvend stemming på de filtrerede tokens og gem det i en ny kolonne 'stemmed'
data['stemmed'] = data['filtered_tokens'].apply(
    lambda x: [ps.stem(w) if not w.startswith('<') else w for w in x]
)

# Liste over alle stemmede ord i alle artikler
all_stemmed_words = [word for sublist in data['stemmed'] for word in sublist]

# Sæt af unikke stemmede ord (vocab) og beregn vocab size
vocab_stemmed = set(all_stemmed_words)
vocab_stemmed_size = len(vocab_stemmed)

# Reduktion efter stopwords i forhold til baseline
reduction_stop = (1 - (filtered_vocab_size / vocab_size)) * 100

# Reduktion efter stemming i forhold til efter stopwords
reduction_stem = (1 - (vocab_stemmed_size / filtered_vocab_size)) * 100

# PRINT RESULTATER 
print(f"\n--- STATISTIK FOR TASK 1 ---")
print(f"Oprindelig Vocab Size: {vocab_size}")
print(f"Vocab Size efter Stopwords: {filtered_vocab_size} (Reduktion: {reduction_stop:.2f}%)")
print(f"Vocab Size efter Stemming: {vocab_stemmed_size} (Reduktion: {reduction_stem:.2f}%)")